# Fossil Looker

---

**Fossil Looker** é um sistema classificador de dinossauros. A intenção é que, baseado em características visíveis em fósseis (e imagens de dinossauros, no geral), qualquer usuário consiga identificar dinossauros através de um funilamento de características.

O sistema é desenvolvido utilizando *Experta*, e para começar é necessário ajustar alguns imports e instalar a biblioteca.

In [43]:
# A lib FrozenDict, que é requisito do Experta, não suporta o Collections do Python 3.12,
# então tenho que fazer essa reatribuição pra corrigir os problemas.
import collections
import collections.abc

collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping
collections.Sequence = collections.abc.Sequence

!pip install experta
from experta import *

O **Fossil Looker** se baseia em um sistema de encadeamento que segue 3 níveis, e cada nível representa um aprofundamento taxonômico das espécies de dinossauro.

- Nível 1: Características vísiveis que determinam fatos gerais sobre os dinossauros, como alimentação, locomoção, etc

- Nível 2: União de características que restringem a taxonomia (subordens e grupos)

- Nível 3: Determinação da espécie de fato

In [44]:
class Dino(Fact):
    pass

## Nível 1. Características Visuais

Aqui temos 4 características básicas e visuais que qualquer pessoa pode identificar, que ajuda a definir os grupos taxônomicos maiores de dinossauros.

- Dentes: Afiados ou Achatados
- Locomoção: Bípede ou Quadrúpede

In [45]:
# ----- Nível 1: Características Visuais -----

class Nivel1(KnowledgeEngine):

  # R1: Dentes Afiados -> Carnívoro
  @Rule(Dino(dentes='afiados'))
  def r1_carnivoro(self):
      print("R1: carnívoro")
      self.declare(Dino(tipo='carnivoro'))

  # R2: Dentes Achatados -> Herbívoro
  @Rule(Dino(dentes='achatados'))
  def r2_herbivoro(self):
      self.log("R2: herbívoro")
      self.declare(Dino(tipo='herbivoro'))

  # R3: Locomoção -> Bípede
  @Rule(Dino(bipede=True))
  def r3_bipede(self):
      self.log("R3: bípede")
      self.declare(Dino(locomocao='bipede'))

  # R4: Locomoção -> Quadrúpede
  @Rule(Dino(quadrupede=True))
  def r4_quadrupede(self):
      self.log("R4: quadrúpede")
      self.declare(Dino(locomocao='quadrupede'))

## Nível 2. Agrupamento Taxonômico

Aqui já começamos a agrupar as características visíveis em combinações, para determinar os grupos possíveis de dinossauros.

- Carnívoro + Bípede -> Terópode
- Herbívoro + Bípede -> Ornitópode
- Herbívoro + Quadrúpede -> Outros Herbívoros (Ceratopsídeos, Tireóforos, Sauropodes, etc)

Perceba que não existem registros de carnívoros quadrúpedes, então resulta em um erro.

In [46]:
# ----- Nível 2: Agrupamento Taxonômico -----

class Nivel2(Nivel1):

  # R5: Carnívoro + Bípede -> Terópode
  @Rule(Dino(tipo='carnivoro'),
        Dino(locomocao='bipede'))
  def r5_teropode(self):
      self.log("R5: harnívoro e bípede => Terópode")
      self.declare(Dino(grupo='teropode'))

  # R6: Carnívoro + Quadrúpede -> NÃO EXISTE
  @Rule(Dino(tipo='carnivoro'),
        Dino(locomocao='quadrupede'))
  def r6_erro_carnivoro_quad(self):
      self.log("R6: ERRO - Não existiram dinossauros carnívoros quadrúpedes.")

  # R7: Herbívoro + Bípede -> Ornitópode
  @Rule(Dino(tipo='herbivoro'),
        Dino(locomocao='bipede'))
  def r7_ornitopode(self):
      self.log("R7: herbívoro e bípede => Ornitopode")
      self.declare(Dino(grupo='ornitopode'))

  # R8: Herbivoro + Quadrupede -> Ornithischia
  @Rule(Dino(tipo='herbivoro'),
        Dino(locomocao='quadrupede'))
  def r8_outros_herb(self):
      self.log("R8: herbívoro e quadrúpede => Outros Herbívoros")
      self.declare(Dino(grupo='outros_herbivoros'))

## Nível 3. Classificação de Gênero/Espécie

No último nível ocorre a distinção da espécie (ou gênero, porque muitos continham espécies com diferenças mínimas), e essa distinção se dá pelo apêndice que o dinossauro contém em seu fóssil. Os apêndices podem ser:

- Chifres
- Apêndice Dorsal (Velas ou Placas conectadas às costas)
- Crista

Perceba que aqui também há um erro, porque não existem registros de dinossauros ornitopodes com chifres.

In [47]:
# ----- Nível 3: Classificação de Gênero/Espécie -----

class FossilLooker(Nivel2):
    def __init__(self):
        super().__init__()
        self.logs = []

    def log(self, msg):
        self.logs.append(msg)
        print(msg)

    # -------------------------
    # TERÓPODES
    # -------------------------

    # R9: Terópode + Chifres -> Ceratosaurus
    @Rule(Dino(grupo='teropode'),
          Dino(chifres=True),
          NOT(Dino(crista=True)),
          NOT(Dino(vela_dorsal=True)))
    def r9_ceratosaurus(self):
        self.log("R9: terópode e chifres => Ceratosaurus")
        self.declare(Dino(especie='ceratosaurus'))

    # R10: Terópode + Dorsal -> Spinosaurus
    @Rule(Dino(grupo='teropode'),
          Dino(vela_dorsal=True),
          NOT(Dino(chifres=True)),
          NOT(Dino(crista=True)))
    def r10_spinosaurus(self):
        self.log("R10: terópode e vela dorsal => Spinosaurus")
        self.declare(Dino(especie='spinosaurus'))

    # R11: Terópode + Crista -> Cryolophosaurus
    @Rule(Dino(grupo='teropode'),
          Dino(crista=True),
          NOT(Dino(chifres=True)),
          NOT(Dino(vela_dorsal=True)))
    def r11_cryolophosaurus(self):
        self.log("R11: terópode e crista => Cryolophosaurus")
        self.declare(Dino(especie='cryolophosaurus'))

    # -------------------------
    # ORNITÓPODES
    # -------------------------

    # R12: Ornitopode + Dorsal -> Ouranosaurus
    @Rule(Dino(grupo='ornitopode'),
          Dino(vela_dorsal=True),
          NOT(Dino(crista=True)),
          NOT(Dino(chifres=True)))
    def r12_ouranosaurus(self):
        self.log("R12: ornitópode e vela dorsal => Ouranosaurus")
        self.declare(Dino(especie='ouranosaurus'))

    # R13: Ornitopode + Crista -> Parasaurolophus
    @Rule(Dino(grupo='ornitopode'),
          Dino(crista=True),
          NOT(Dino(vela_dorsal=True)),
          NOT(Dino(chifres=True)))
    def r13_parasaurolophus(self):
        self.log("R13: ornitópode e crista => Parasaurolophus")
        self.declare(Dino(especie='parasaurolophus'))

    # R14: Ornitopode + Chifres -> NÃO EXISTE
    @Rule(Dino(grupo='ornitopode'),
          Dino(chifres=True))
    def r14_erro_ornitopode_chifre(self):
        self.log("R14: ERRO - ornitópodes não possuem chifres")

    # -------------------------
    # OUTROS HERBÍVOROS
    # -------------------------

    # R15: Outros Herbívoros + Chifres -> Triceratops
    @Rule(Dino(grupo='outros_herbivoros'),
          Dino(chifres=True),
          NOT(Dino(crista=True)),
          NOT(Dino(placas_dorsais=True)))
    def r15_triceratops(self):
        self.log("R15: outro herbívoro e chifres => Triceratops")
        self.declare(Dino(especie='triceratops'))

    # R16: Outros Herbívoros + Dorsal -> Stegosaurus
    @Rule(Dino(grupo='outros_herbivoros'),
          Dino(placas_dorsais=True),
          NOT(Dino(chifres=True)),
          NOT(Dino(crista=True)))
    def r16_stegosaurus(self):
        self.log("R16: outro herbívoro e placas dorsais => Stegosaurus")
        self.declare(Dino(especie='stegosaurus'))

    # R17: Outros Herbívoros + Crista -> Pachyrhinosaurus
    @Rule(Dino(grupo='outros_herbivoros'),
          Dino(crista=True),
          NOT(Dino(chifres=True)),
          NOT(Dino(placas_dorsais=True)))
    def r17_pachyrhinosaurus(self):
        self.log("R17: outro herbívoro e crista => Pachyrhinosaurus")
        self.declare(Dino(especie='pachyrhinosaurus'))

## Casos de teste

Vamos testar 3 casos aqui.

In [48]:
# Primeiro, instanciando a engine para rodar os testes
engine = FossilLooker()

In [49]:
# Caso 1: Spinosaurus
engine.reset()
engine.declare(Dino(
    dentes='afiados',
    bipede=True,
    vela_dorsal=True
))
engine.run()

R1: carnívoro
R3: bípede
R5: harnívoro e bípede => Terópode
R10: terópode e vela dorsal => Spinosaurus


In [50]:
# Caso 2: Parasaurolophus
engine.reset()
engine.declare(Dino(
    dentes='achatados',
    bipede=True,
    crista=True
))
engine.run()

R3: bípede
R2: herbívoro
R7: herbívoro e bípede => Ornitopode
R13: ornitópode e crista => Parasaurolophus


In [51]:
# Caso 3: Inválido (Carnívoro Quadrúpede)
engine.reset()
engine.declare(Dino(
    dentes='afiados',
    quadrupede=True
))
engine.run()


R1: carnívoro
R4: quadrúpede
R6: ERRO - Não existiram dinossauros carnívoros quadrúpedes.
